# 6장 예제 — 6.21 실전 시계열 ③ 시계열 예측 — 1분 뒤 온도와 금형 교체 시점 (실습 45분) `필수`

- **교재 출처**: manuscript/61_ch6_최적화.md
- **포함 소절**: 왜 예측인가 — 이상탐지와의 관계 / 지연 특징 만들기와 시계열 교차검증 / 예측 잔차로 이상 잡기 / 예측 잔차가 못 잡는 것 — 완만한 드리프트 / 금형 교체 시점 예측 — 추세 외삽 / 정리 — 예측 세 줄
- **실행 방법**: example 폴더에서  python "6장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **실행 상태**: 단독 실행 확인됨(이 파일만 돌려도 끝까지 실행된다).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

### 지연 특징 만들기와 시계열 교차검증

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

log = pd.read_csv("data/plc/plc_line_log.csv", parse_dates=["timestamp"], index_col="timestamp")
events = pd.read_csv("data/plc/plc_events.csv")

y = log["EC_C_DR2_TEMP_PV"] - log["EC_C_DR2_TEMP_SV"]        # 예측 대상도 편차(PV-SV)다
feat = pd.DataFrame({f"lag{k}": y.shift(k) for k in (1, 2, 3, 5, 10)})
feat["이동평균10"] = y.shift(1).rolling(10).mean()
data = feat.join(y.rename("y")).dropna()
print(data.shape)
print(data.head(3).round(2).to_string())

# 교재 실행 결과 ------------------------------------------------------
#   (10070, 7)
#                        lag1  lag2  lag3  lag5  lag10  이동평균10     y
#   timestamp                                                       
#   2026-09-01 00:10:00  1.42  1.43  0.73  0.09  -0.31    0.29  1.14
#   2026-09-01 00:11:00  1.14  1.42  1.43  0.32  -0.30    0.44  0.82
#   2026-09-01 00:12:00  0.82  1.14  1.42  0.73   0.00    0.55  1.36

In [ ]:
학습 = data.loc[:"2026-09-04 23:59"]                    # 1~4일차로 학습·검증, 5~7일차는 봉인
Xtr, ytr = 학습.drop(columns="y"), 학습["y"]
tscv = TimeSeriesSplit(n_splits=5)
결과 = {"naive(lag1)": [], "선형 회귀": [], "랜덤 포레스트": []}
for tr, va in tscv.split(Xtr):
    결과["naive(lag1)"].append(mean_absolute_error(ytr.iloc[va], Xtr["lag1"].iloc[va]))
    lr = LinearRegression().fit(Xtr.iloc[tr], ytr.iloc[tr])
    결과["선형 회귀"].append(mean_absolute_error(ytr.iloc[va], lr.predict(Xtr.iloc[va])))
    rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf.fit(Xtr.iloc[tr], ytr.iloc[tr])
    결과["랜덤 포레스트"].append(mean_absolute_error(ytr.iloc[va], rf.predict(Xtr.iloc[va])))
print(pd.DataFrame(결과).round(3).assign(fold=range(1, 6)).set_index("fold").T.to_string())
print("평균 MAE(℃):", {k: round(float(np.mean(v)), 3) for k, v in 결과.items()})

# 교재 실행 결과 ------------------------------------------------------
#   fold             1      2      3      4      5
#   naive(lag1)  0.271  0.275  0.287  0.272  0.277
#   선형 회귀        0.257  0.266  0.276  0.264  0.264
#   랜덤 포레스트      0.272  0.274  0.306  0.272  0.274
#   평균 MAE(℃): {'naive(lag1)': 0.276, '선형 회귀': 0.265, '랜덤 포레스트': 0.28}

### 예측 잔차로 이상 잡기

In [ ]:
lr = LinearRegression().fit(Xtr, ytr)
잔차_학습 = ytr - lr.predict(Xtr)
σ = 잔차_학습.std()
print("학습 기간 잔차 표준편차:", round(σ, 3), "℃ → 임계 ±", round(3 * σ, 2), "℃")

시험 = data.loc["2026-09-05":]
예측 = pd.Series(lr.predict(시험.drop(columns="y")), index=시험.index)
잔차 = 시험["y"] - 예측
검출 = 잔차.abs() > 3 * σ
print("5~7일차 예측 MAE:", round(mean_absolute_error(시험["y"], 예측), 3), "℃")
print("잔차 이상 검출:", int(검출.sum()), "분")
print(잔차[검출].round(2).to_string())

# 교재 실행 결과 ------------------------------------------------------
#   학습 기간 잔차 표준편차: 0.369 ℃ → 임계 ± 1.11 ℃
#   5~7일차 예측 MAE: 0.27 ℃
#   잔차 이상 검출: 3 분
#   timestamp
#   2026-09-05 03:10:00    7.12
#   2026-09-05 03:11:00    1.47
#   2026-09-05 03:13:00   -5.91

In [ ]:
구간 = slice("2026-09-05 02:40", "2026-09-05 03:40")
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
시험.loc[구간, "y"].plot(ax=axes[0], lw=1.2, label="실측 편차")
예측.loc[구간].plot(ax=axes[0], lw=1.2, ls="--", label="1분 뒤 예측(선형 회귀)")
axes[0].set_ylabel("건조로 2구간\nPV-SV (℃)"); axes[0].legend(loc="upper left")
잔차.loc[구간].plot(ax=axes[1], lw=1.0, color="gray")
잔차.loc[구간][검출.loc[구간]].plot(ax=axes[1], style="r.", ms=8, label="|잔차| > 3σ")
axes[1].axhline(3 * σ, color="r", ls=":"); axes[1].axhline(-3 * σ, color="r", ls=":")
axes[1].set_ylabel("잔차 = 실측 - 예측(℃)"); axes[1].legend(loc="upper left")
plt.tight_layout(); plt.savefig("fig_6_42_plc_forecast_resid.png", dpi=150); plt.close()

### 예측 잔차가 못 잡는 것 — 완만한 드리프트

In [ ]:
y3 = log["EC_C_DR3_TEMP_PV"] - log["EC_C_DR3_TEMP_SV"]
f3 = pd.DataFrame({f"lag{k}": y3.shift(k) for k in (1, 2, 3, 5, 10)})
f3["이동평균10"] = y3.shift(1).rolling(10).mean()
d3 = f3.join(y3.rename("y")).dropna()
tr3, te3 = d3.loc[:"2026-09-04 23:59"], d3.loc["2026-09-05":]
lr3 = LinearRegression().fit(tr3.drop(columns="y"), tr3["y"])
σ3 = (tr3["y"] - lr3.predict(tr3.drop(columns="y"))).std()
잔차3 = te3["y"] - lr3.predict(te3.drop(columns="y"))
C3구간 = slice("2026-09-06 09:00", "2026-09-06 15:00")
print("C3 구간 잔차 절댓값 최대:", round(잔차3.loc[C3구간].abs().max(), 2), "℃ | 임계 3σ =", round(3 * σ3, 2),
      "℃ → 검출", int((잔차3.loc[C3구간].abs() > 3 * σ3).sum()), "분 /", len(잔차3.loc[C3구간]))

편차3 = log["EC_C_DR3_TEMP_PV"] - log["EC_C_DR3_TEMP_SV"]
편차3_60 = 편차3.rolling("60min").mean()                     # 편차의 60분 이동평균 — 기준은 SV(고정)
기준sd = 편차3_60.loc[:"2026-09-02 23:59"].std()
검출3 = 편차3_60.abs() > 3 * 기준sd
print("편차 60분 이동평균 |·| > 3σ(참조):", int(검출3.loc[C3구간].sum()), "분 검출, 첫 검출",
      검출3.loc[C3구간][검출3.loc[C3구간]].index.min())

# 교재 실행 결과 ------------------------------------------------------
#   C3 구간 잔차 절댓값 최대: 1.41 ℃ | 임계 3σ = 1.0 ℃ → 검출 4 분 / 361
#   편차 60분 이동평균 |·| > 3σ(참조): 218 분 검출, 첫 검출 2026-09-06 11:23:00

### 금형 교체 시점 예측 — 추세 외삽

In [ ]:
횟수 = log["EC_N_금형횟수_현재값"]
경보 = int(log["EC_N_금형횟수_경보_설정"].iloc[-1])
교체시각 = 횟수.index[횟수.diff() < 0]                     # 횟수가 줄어든 순간 = 교체
print("금형 교체 시각:", list(교체시각.strftime("%m-%d %H:%M")))
최근 = 횟수.loc[교체시각[-1]:]                             # 마지막 교체 이후만 사용
print("마지막 교체 이후 경과:", round((최근.index[-1] - 최근.index[0]).total_seconds() / 3600, 1),
      "시간 | 현재 횟수:", int(최근.iloc[-1]))

경과분 = ((최근.index - 최근.index[0]).total_seconds() / 60).to_numpy().reshape(-1, 1)
lr = LinearRegression().fit(경과분, 최근.to_numpy())
기울기 = lr.coef_[0]                                       # 회/분 (정지 시간이 섞인 평균 속도)
남은분 = (경보 - 최근.iloc[-1]) / 기울기
도달예상 = 최근.index[-1] + pd.Timedelta(minutes=남은분)
print("평균 속도:", round(기울기, 1), "회/분 →", round(기울기 * 1440), "회/일")
print("경보 도달 예상:", 도달예상.strftime("%Y-%m-%d %H:%M"), "(", round(남은분 / 60, 1), "시간 뒤 )")

# 교재 실행 결과 ------------------------------------------------------
#   금형 교체 시각: ['09-01 01:30', '09-04 23:04']
#   마지막 교체 이후 경과: 72.9 시간 | 현재 횟수: 468070
#   평균 속도: 106.9 회/분 → 153882 회/일
#   경보 도달 예상: 2026-09-08 20:33 ( 20.6 시간 뒤 )

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
횟수.plot(ax=ax, lw=1, x_compat=True, label="금형 타발 누적 횟수")   # x_compat: 날짜 축을 matplotlib 방식으로
미래 = pd.date_range(최근.index[0], 도달예상, freq="60min")
미래분 = ((미래 - 최근.index[0]).total_seconds() / 60).to_numpy().reshape(-1, 1)
ax.plot(미래, lr.predict(미래분), "r--", label="추세 외삽")
ax.axhline(경보, color="k", ls=":", label="경보 설정 600,000")
ax.axvline(도달예상, color="r", ls=":")
ax.text(도달예상, 경보 * 0.45, "도달 예상\n" + 도달예상.strftime("%m-%d %H:%M"), color="r", ha="right")
ax.set_ylabel("횟수"); ax.legend(loc="upper left")
plt.savefig("fig_6_43_plc_die_forecast.png", dpi=150, bbox_inches="tight"); plt.close()